# OEP Curve — UK Flood Cat Model

The **Occurrence Exceedance Probability (OEP)** curve shows:
- *x-axis*: Return period in years (log scale, 2–1000yr)
- *y-axis*: Gross annual loss in £bn corresponding to that return period

Key output uses:
- **AAL** (average annual loss): base flood loading for pricing
- **1-in-100**: Solvency II internal model benchmark
- **1-in-200**: Solvency II SCR capital requirement
- **Known events**: ABI-published insured losses for 13 UK flood events (2000–2024)

The 2007 UK floods (£3.2bn insured) is the single most important calibration
point. A well-calibrated model places the 1-in-75 year loss between £2.5bn and £4.5bn.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..').resolve()))

from src.metrics.return_periods import (
    build_loss_exceedance_curve,
    compute_aal,
    get_return_period_losses,
)

RESULTS_DIR = Path('../outputs/results')
PLOTS_DIR = Path('../outputs/plots')
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print('Packages loaded successfully')

## 1. Load LEC and compute bootstrap uncertainty bands

The LEC parquet contains 10,000 simulated events. Bootstrap uncertainty
reflects parameter uncertainty in the GEV fitting + Monte Carlo sampling.

In [ ]:
# Load LEC
lec_df = pd.read_parquet(RESULTS_DIR / 'loss_exceedance_curve.parquet')

# Filter to plottable range (T = 2 to 1000 years)
plot_df = lec_df[
    (lec_df['return_period_years'] >= 2) &
    (lec_df['return_period_years'] <= 1000)
].copy()

print(f'LEC records: {len(lec_df):,} events')
print(f'Plot range: T={plot_df["return_period_years"].min():.1f}–{plot_df["return_period_years"].max():.0f} years')
print(f'Loss range: £{plot_df["loss_gbp"].min()/1e9:.2f}bn–£{plot_df["loss_gbp"].max()/1e9:.2f}bn')

# AAL and key return period losses
aal = compute_aal(lec_df)
rp_losses = get_return_period_losses(lec_df, [10, 50, 100, 200, 500])

print(f'\nAAL:         £{aal/1e6:,.0f}m')
for T, loss in rp_losses.items():
    print(f'1-in-{T:<5} yr: £{loss/1e9:.2f}bn')

In [ ]:
# Bootstrap 90% uncertainty bands by resampling event rates
N_BOOTSTRAP = 500
np.random.seed(42)

# Target return periods for the band
T_targets = np.logspace(np.log10(2), np.log10(1000), 80)
boot_losses = np.zeros((N_BOOTSTRAP, len(T_targets)))

losses_arr = lec_df['loss_gbp'].values
rates_arr = lec_df['exceedance_rate'].diff().abs().fillna(lec_df['exceedance_rate'].iloc[-1]).values
# Ensure rates are positive
rates_arr = np.abs(rates_arr)
rates_arr[rates_arr <= 0] = rates_arr[rates_arr > 0].min()

for i in range(N_BOOTSTRAP):
    # Resample events with replacement
    idx = np.random.choice(len(losses_arr), size=len(losses_arr), replace=True)
    boot_losses_i = losses_arr[idx]
    boot_rates_i = rates_arr[idx]
    
    # Sort by descending loss and compute cumulative rate
    order = np.argsort(boot_losses_i)[::-1]
    sorted_losses = boot_losses_i[order]
    cum_rates = np.cumsum(boot_rates_i[order])
    boot_T = 1.0 / np.maximum(cum_rates, 1e-10)
    
    # Interpolate to target return periods
    boot_losses[i] = np.interp(T_targets, boot_T[::-1], sorted_losses[::-1])

ci_lower = np.percentile(boot_losses, 5, axis=0)
ci_upper = np.percentile(boot_losses, 95, axis=0)
ci_median = np.percentile(boot_losses, 50, axis=0)
print(f'Bootstrap complete: {N_BOOTSTRAP} resamples')

## 2. Load ABI validation events

In [ ]:
events_df = pd.read_parquet(Path('../data/raw/abi_events/uk_flood_events.parquet'))
# Only high/medium confidence events for plotting
events_plot = events_df[events_df['confidence'].isin(['high', 'medium'])].copy()

# Map each event's insured loss to a return period from the LEC
events_plot['loss_gbp'] = events_plot['insured_loss_gbp_m'] * 1e6
events_plot['implied_T'] = events_plot['loss_gbp'].apply(
    lambda loss: np.interp(
        loss,
        lec_df['loss_gbp'].values[::-1],
        lec_df['return_period_years'].values[::-1],
    )
)

print('Validation events:')
print(events_plot[['name','year','insured_loss_gbp_m','implied_T']].to_string(index=False))

## 3. Plotly interactive OEP curve

In [ ]:
fig = go.Figure()

# 90% CI band
fig.add_trace(go.Scatter(
    x=np.concatenate([T_targets, T_targets[::-1]]),
    y=np.concatenate([ci_upper / 1e9, ci_lower[::-1] / 1e9]),
    fill='toself',
    fillcolor='rgba(70,130,180,0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='90% bootstrap CI',
))

# Main LEC curve
fig.add_trace(go.Scatter(
    x=plot_df['return_period_years'],
    y=plot_df['loss_gbp'] / 1e9,
    name='OEP (modelled)',
    line=dict(color='steelblue', width=2.5),
))

# ABI event benchmarks
marker_colors = {'high': '#d62728', 'medium': '#ff7f0e'}
for _, ev in events_plot.iterrows():
    fig.add_trace(go.Scatter(
        x=[ev['implied_T']],
        y=[ev['loss_gbp'] / 1e9],
        mode='markers+text',
        name=f"{ev['name'][:25]} ({ev['year']})",
        marker=dict(
            size=10,
            color=marker_colors.get(ev['confidence'], 'grey'),
            symbol='circle',
            line=dict(width=1.5, color='white'),
        ),
        text=[f"{ev['name'][:20]}<br>£{ev['insured_loss_gbp_m']/1e3:.1f}bn"],
        textposition='top right',
        textfont=dict(size=9),
        showlegend=False,
    ))

# Horizontal reference lines
ref_lines = {
    'AAL': aal / 1e9,
    '1-in-100': rp_losses[100] / 1e9,
    '1-in-200': rp_losses[200] / 1e9,
}
for label, val in ref_lines.items():
    fig.add_hline(
        y=val, line_dash='dot', line_color='grey', line_width=1,
        annotation_text=f'{label}: £{val:.1f}bn',
        annotation_position='right',
        annotation_font_size=10,
    )

fig.update_layout(
    title=dict(
        text='UK Residential Flood OEP Curve — Gross Loss vs Return Period',
        font=dict(size=16),
    ),
    xaxis=dict(
        title='Return Period (years)',
        type='log',
        range=[np.log10(2), np.log10(1000)],
        tickvals=[2, 5, 10, 20, 50, 100, 200, 500, 1000],
        ticktext=['2', '5', '10', '20', '50', '100', '200', '500', '1000'],
    ),
    yaxis=dict(title='Gross Loss (£bn)', rangemode='tozero'),
    legend=dict(x=0.02, y=0.98, font=dict(size=10)),
    template='plotly_white',
    height=550,
)

fig.write_html(str(PLOTS_DIR / 'oep_curve.html'))
print(f'Saved interactive OEP → {PLOTS_DIR}/oep_curve.html')
fig.show()

## 4. Matplotlib publication-quality OEP curve (300dpi PNG)

In [ ]:
fig_mpl, ax = plt.subplots(figsize=(10, 6))

# CI band
ax.fill_between(
    T_targets, ci_lower / 1e9, ci_upper / 1e9,
    alpha=0.2, color='steelblue', label='90% bootstrap CI'
)

# LEC
ax.plot(
    plot_df['return_period_years'], plot_df['loss_gbp'] / 1e9,
    color='steelblue', linewidth=2.5, label='OEP (modelled)'
)

# ABI event markers
event_colors = {'high': '#d62728', 'medium': '#ff7f0e'}
for _, ev in events_plot.iterrows():
    c = event_colors.get(ev['confidence'], 'grey')
    ax.scatter(ev['implied_T'], ev['loss_gbp'] / 1e9, color=c, s=60, zorder=5, edgecolors='white', linewidths=0.8)
    ax.annotate(
        f"{ev['year']}",
        (ev['implied_T'], ev['loss_gbp'] / 1e9),
        textcoords='offset points', xytext=(5, 3),
        fontsize=8, color=c,
    )

# Reference lines
ax.axhline(aal / 1e9, color='grey', linestyle=':', linewidth=1, label=f'AAL: £{aal/1e9:.2f}bn')
ax.axhline(rp_losses[200] / 1e9, color='darkgrey', linestyle='--', linewidth=1,
           label=f'1-in-200yr: £{rp_losses[200]/1e9:.1f}bn')

ax.set_xscale('log')
ax.set_xlabel('Return Period (years)', fontsize=12)
ax.set_ylabel('Gross Loss (£bn)', fontsize=12)
ax.set_title('UK Residential Flood OEP Curve', fontsize=14, fontweight='bold')
ax.set_xlim(2, 1000)
ax.set_ylim(bottom=0)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%g'))
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, which='major', alpha=0.3)
ax.grid(True, which='minor', alpha=0.1)

# ABI event legend patches
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#d62728', markersize=7, label='ABI event (high confidence)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#ff7f0e', markersize=7, label='ABI event (medium confidence)'),
]
ax.legend(handles=ax.get_legend_handles_labels()[0] + legend_elements,
          labels=ax.get_legend_handles_labels()[1] + [e.get_label() for e in legend_elements],
          fontsize=9, loc='upper left')

plt.tight_layout()
png_path = PLOTS_DIR / 'oep_curve.png'
fig_mpl.savefig(png_path, dpi=300, bbox_inches='tight')
print(f'Saved publication PNG → {png_path}')
plt.show()